# Dataset formats

> A registry of on-disk dataset formats, each with a streaming writer and a reader.

In [ ]:
#| default_exp data.format

In [ ]:
#| hide
from nbdev.showdoc import *

Same registry as stable-worldmodel: a `Format` ties a name to a `detect` check, a writer
and a reader. New formats only need a `Format` subclass decorated with `@register_format`.

An **episode** handed to a writer is a dict ``{column: array}``, every column with one row per
step (see `World.collect`).

In [ ]:
#| export
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Protocol, runtime_checkable

In [ ]:
#| export
FORMATS: dict[str, type[Format]] = {}

WRITE_MODES = ('append', 'overwrite', 'error')
"""
Writer modes shared by all formats: ``append`` extends an existing dataset (default),
``overwrite`` replaces it, ``error`` raises :class:`FileExistsError` if it exists.
"""


EPISODE_DATA_KEY = '_episode_data'
"""
Reserved key of an episode dict holding its episode-scoped values (one per episode, not per
step). Writers of formats with ``supports_episode_data`` split it out with
:func:`split_episode_data`; the others never receive it (``convert`` / ``merge`` drop it first).
"""


def split_episode_data(ep_data: dict) -> tuple[dict, dict]:
    "``(per-step columns, episode data)`` of an episode dict, without changing it."
    if EPISODE_DATA_KEY not in ep_data:
        return ep_data, {}
    return {k: v for k, v in ep_data.items() if k != EPISODE_DATA_KEY}, dict(ep_data[EPISODE_DATA_KEY] or {})


def validate_write_mode(mode: str) -> str:
    if mode not in WRITE_MODES:
        raise ValueError(f"write mode must be one of {WRITE_MODES}, got {mode!r}")
    return mode


def register_format(cls: type[Format]) -> type[Format]:
    "Class decorator registering a :class:`Format` under its `name`."
    name = getattr(cls, 'name', None)
    if not name:
        raise ValueError(f"{cls.__name__} must set a non-empty `name` class attribute")
    if name in FORMATS and FORMATS[name].__qualname__ != cls.__qualname__:   # same class re-defined: replace
        raise ValueError(f"format {name!r} is already registered")
    FORMATS[name] = cls
    return cls


def list_formats() -> list[str]:
    return list(FORMATS)


def get_format(name: str) -> type[Format]:
    try:
        return FORMATS[name]
    except KeyError:
        raise ValueError(f"unknown format {name!r}; available: {list_formats()}") from None


def detect_format(path: str | Path) -> type[Format] | None:
    "The first registered format whose ``detect`` matches `path`, else None."
    for fmt in FORMATS.values():
        if fmt.detect(path):
            return fmt
    return None


class Format:
    "An on-disk dataset format: subclasses set `name`, implement `detect` and their reader / writer."
    name: str = ''
    #: whether the writer stores episode-scoped data (:data:`EPISODE_DATA_KEY`)
    supports_episode_data: bool = False

    @classmethod
    def detect(cls, path) -> bool:
        raise NotImplementedError(f"{cls.__name__}.detect must be implemented")

    @classmethod
    def open_reader(cls, path, **kwargs):
        raise NotImplementedError(f"format {cls.name or cls.__name__!r} does not support reading")

    @classmethod
    def open_writer(cls, path, **kwargs) -> Writer:
        "A streaming :class:`Writer`; built-in writers take a ``mode`` from :data:`WRITE_MODES`."
        raise NotImplementedError(f"format {cls.name or cls.__name__!r} does not support writing")


@runtime_checkable
class Writer(Protocol):
    "Streaming writer: append episodes inside a ``with`` block."
    def __enter__(self) -> Writer: ...
    def __exit__(self, *exc) -> None: ...
    def write_episode(self, ep_data: dict) -> None: ...
    def write_episodes(self, episodes: Iterable[dict]) -> None: ...

### Tests

In [ ]:
from fastcore.test import test_fail
@register_format
class Dummy(Format): name = 'dummy'
@register_format
class Dummy(Format): name = 'dummy'       # re-defining the same class (notebooks, reloads) is fine
assert get_format('dummy') is Dummy and 'dummy' in list_formats()
def clash():
    @register_format
    class Other(Format): name = 'dummy'
test_fail(clash, contains='already registered')
del FORMATS['dummy']

steps, extra = split_episode_data({'pixels': [1, 2], EPISODE_DATA_KEY: {'task': 'a'}})
assert steps == {'pixels': [1, 2]} and extra == {'task': 'a'}
plain = {'pixels': [1]}
assert split_episode_data(plain) == (plain, {}) and not Format.supports_episode_data


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()